In [ ]:
# %% [code]
import json, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.neural_network import MLPRegressor

# Optional XGBoost
try:
    import xgboost as xgb
except ImportError:
    xgb = None

SEED = 42
np.random.seed(SEED)
EPS = 1e-9


In [ ]:
# %% [code]
# =========================
# Config (speed vs MAE)
# =========================
FAST_MODE = True      # κράτα True για λογικό runtime
USE_GPU = False       # CPU για σταθερότητα (βάλε True μόνο αν έχεις σταθερό CUDA setup)

# CV setup
TUNE_SPLITS = 3       # tuning folds
FINAL_SPLITS = 5      # final folds (μόνο για top2)

# Trials (random search)
# Αν θες λίγο καλύτερο MAE και δέχεσαι λίγα λεπτά παραπάνω: αύξησε TRIALS_XGB σε 18-22
TRIALS_XGB = 12 if FAST_MODE else 24
TRIALS_RF  = 5  if FAST_MODE else 10
TRIALS_ET  = 5  if FAST_MODE else 10
TRIALS_MLP = 4  if FAST_MODE else 8

# XGB bagging seeds (λίγοι για να μην αργεί)
XGB_OOF_SEEDS = [SEED, SEED + 1337] if FAST_MODE else [SEED, SEED + 1337, SEED + 2024]

# Early stopping rounds for XGB
EARLY_STOPPING_ROUNDS = 30 if FAST_MODE else 60

# Weighting strategy
WEIGHT_MODE = "sqrt"   # none/raw/sqrt/clip10


In [ ]:
# %% [code]
def load_data():
    df = pd.read_csv("c13k_selections.csv")
    with open("c13k_problems.json", "r") as f:
        problems_dict = json.load(f)
    return df, problems_dict

def clip01(a):
    return np.clip(np.asarray(a, dtype=float), 0.0, 1.0)

def metrics(y_true, y_pred):
    y_pred = clip01(y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

def safe_div(a, b):
    return float(a / (b + EPS))

def soft_ratio(a, b, scale=1.0):
    return float(a / (abs(b) + scale))


In [ ]:
# %% [code]
def get_gamble_stats(outcomes):
    probs = np.array([o[0] for o in outcomes], dtype=float)
    probs = probs / (probs.sum() + EPS)
    pays  = np.array([o[1] for o in outcomes], dtype=float)

    ev = float(np.sum(probs * pays))
    var = float(np.sum(probs * (pays - ev) ** 2))
    sd = float(np.sqrt(var))

    mn = float(np.min(pays))
    mx = float(np.max(pays))
    rng = float(mx - mn)

    p_gain = float(np.sum(probs[pays > 0])) if np.any(pays > 0) else 0.0
    p_loss = float(np.sum(probs[pays < 0])) if np.any(pays < 0) else 0.0
    p_zero = float(np.sum(probs[pays == 0])) if np.any(pays == 0) else 0.0

    ev_gain = float(np.sum(probs[pays > 0] * pays[pays > 0])) if np.any(pays > 0) else 0.0
    ev_loss = float(np.sum(probs[pays < 0] * pays[pays < 0])) if np.any(pays < 0) else 0.0

    mean_gain = safe_div(ev_gain, p_gain) if p_gain > 0 else 0.0
    mean_loss = safe_div(ev_loss, p_loss) if p_loss > 0 else 0.0

    abs_pays = np.abs(pays)
    mean_abs = float(np.sum(probs * abs_pays))
    max_gain = float(np.max(pays[pays > 0])) if np.any(pays > 0) else 0.0
    max_loss = float(np.min(pays[pays < 0])) if np.any(pays < 0) else 0.0

    if sd > 0:
        z = (pays - ev) / sd
        skew = float(np.sum(probs * z**3))
        kurt = float(np.sum(probs * z**4))
    else:
        skew = 0.0
        kurt = 0.0

    pclip = np.clip(probs, EPS, 1.0)
    ent = float(-np.sum(pclip * np.log(pclip)))

    return {
        "EV": ev, "SD": sd, "Var": var,
        "Min": mn, "Max": mx, "Range": rng,
        "P_Gain": p_gain, "P_Loss": p_loss, "P_Zero": p_zero,
        "EV_Gain": ev_gain, "EV_Loss": ev_loss,
        "Mean_Gain": mean_gain, "Mean_Loss": mean_loss,
        "Mean_Abs": mean_abs, "Max_Gain": max_gain, "Max_Loss": max_loss,
        "Skew": skew, "Kurt": kurt,
        "Entropy": ent,
        "Num_Out": float(len(outcomes)),
    }

def psych_ev(outcomes, alpha=0.88, gamma=0.65):
    probs = np.array([o[0] for o in outcomes], dtype=float)
    probs = probs / (probs.sum() + EPS)
    pays  = np.array([o[1] for o in outcomes], dtype=float)

    subj_pays = np.sign(pays) * (np.abs(pays) ** alpha)
    probs_clipped = np.clip(probs, EPS, 1.0)
    weighted_probs = np.exp(-(-np.log(probs_clipped)) ** gamma)

    return float(np.sum(weighted_probs * subj_pays))

def prob_B_better_than_A(outA, outB):
    pA = np.array([o[0] for o in outA], dtype=float)
    pA = pA / (pA.sum() + EPS)
    xA = np.array([o[1] for o in outA], dtype=float)

    pB = np.array([o[0] for o in outB], dtype=float)
    pB = pB / (pB.sum() + EPS)
    xB = np.array([o[1] for o in outB], dtype=float)

    mat = (xB[:, None] > xA[None, :]).astype(float)
    return float(np.sum((pB[:, None] * pA[None, :]) * mat))


In [ ]:
# %% [code]
def build_dataset(df, problems_dict):
    raw_cols = ["Ha","La","pHa","Hb","Lb","pHb"]
    have_raw = all(c in df.columns for c in raw_cols)
    have_std = "bRate_std" in df.columns
    has_n = "n" in df.columns

    feats, y, w, groups = [], [], [], []

    stat_keys = [
        "EV","SD","Var","Min","Max","Range",
        "P_Gain","P_Loss","P_Zero",
        "EV_Gain","EV_Loss","Mean_Gain","Mean_Loss",
        "Mean_Abs","Max_Gain","Max_Loss",
        "Skew","Kurt","Entropy","Num_Out"
    ]

    problem_cache = {}

    for row in df.itertuples(index=False):
        pid = str(row.Problem)
        base = problem_cache.get(pid)

        if base is None:
            prob = problems_dict.get(pid)
            if prob is None:
                continue

            outA = prob["A"]
            outB = prob["B"]

            sA = get_gamble_stats(outA)
            sB = get_gamble_stats(outB)

            peA  = psych_ev(outA, alpha=0.88, gamma=0.65)
            peB  = psych_ev(outB, alpha=0.88, gamma=0.65)
            peA2 = psych_ev(outA, alpha=0.70, gamma=0.90)
            peB2 = psych_ev(outB, alpha=0.70, gamma=0.90)

            base = {}
            for k in stat_keys:
                base[f"{k}_A"] = sA[k]
                base[f"{k}_B"] = sB[k]
                base[f"{k}_Diff"] = sB[k] - sA[k]

            base["Psych_EV_A"] = peA
            base["Psych_EV_B"] = peB
            base["Psych_EV_Diff"] = peB - peA

            base["Psych_EV2_A"] = peA2
            base["Psych_EV2_B"] = peB2
            base["Psych_EV2_Diff"] = peB2 - peA2

            base["CV_A"] = safe_div(sA["SD"], abs(sA["EV"]) + 1e-6)
            base["CV_B"] = safe_div(sB["SD"], abs(sB["EV"]) + 1e-6)
            base["CV_Diff"] = base["CV_B"] - base["CV_A"]

            # Soft ratios (stable)
            base["EV_Ratio"]      = soft_ratio(sB["EV"],      sA["EV"],      scale=1.0)
            base["SD_Ratio"]      = soft_ratio(sB["SD"],      sA["SD"],      scale=1.0)
            base["Range_Ratio"]   = soft_ratio(sB["Range"],   sA["Range"],   scale=1.0)
            base["P_Gain_Ratio"]  = soft_ratio(sB["P_Gain"],  sA["P_Gain"],  scale=1.0)
            base["P_Loss_Ratio"]  = soft_ratio(sB["P_Loss"],  sA["P_Loss"],  scale=1.0)
            base["Entropy_Ratio"] = soft_ratio(sB["Entropy"], sA["Entropy"], scale=1.0)

            base["P_B_better_A"] = prob_B_better_than_A(outA, outB)

            problem_cache[pid] = base

        rec = base.copy()

        # Design vars from CSV
        rec.update({
            "Amb": int(bool(row.Amb)),
            "Corr": int(row.Corr),
            "Feedback": int(bool(row.Feedback)),
            "LotShapeB": int(row.LotShapeB),
            "LotNumB": int(row.LotNumB),
            "Block": int(row.Block),
        })

        # Raw lottery params (if present) + raw EVs
        if have_raw:
            rec.update({
                "Ha": float(row.Ha),
                "La": float(row.La),
                "pHa": float(row.pHa),
                "Hb": float(row.Hb),
                "Lb": float(row.Lb),
                "pHb": float(row.pHb),
            })
            rec["EV_A_raw"] = float(row.pHa * row.Ha + (1 - row.pHa) * row.La)
            rec["EV_B_raw"] = float(row.pHb * row.Hb + (1 - row.pHb) * row.Lb)
            rec["EV_raw_Diff"] = rec["EV_B_raw"] - rec["EV_A_raw"]

        # Interactions
        rec["Amb_x_SD_Diff"] = rec["Amb"] * rec["SD_Diff"]
        rec["Amb_x_Range_Diff"] = rec["Amb"] * rec["Range_Diff"]
        rec["Feedback_x_EV_Diff"] = rec["Feedback"] * rec["EV_Diff"]
        rec["Feedback_x_PGain_Diff"] = rec["Feedback"] * rec["P_Gain_Diff"]

        # Absolute diffs (magnitude)
        rec["EV_Diff_Abs"] = abs(rec["EV_Diff"])
        rec["SD_Diff_Abs"] = abs(rec["SD_Diff"])
        rec["Entropy_Diff_Abs"] = abs(rec["Entropy_Diff"])
        rec["Psych_EV_Diff_Abs"] = abs(rec["Psych_EV_Diff"])

        # Weights
        n = float(row.n) if has_n else 1.0
        if have_std:
            std = float(row.bRate_std)
            noise_factor = 1.0 / (std*std + 1e-4)
            noise_factor = min(noise_factor, 50.0)
            w.append(n * noise_factor)
        else:
            w.append(n)

        feats.append(rec)
        y.append(float(row.bRate))
        groups.append(pid)

    X = pd.DataFrame(feats).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    y = np.asarray(y, dtype=float)
    w = np.asarray(w, dtype=float)
    groups = np.asarray(groups)
    return X, y, w, groups


In [ ]:
# %% [code]
df, problems_dict = load_data()
X, y, w_raw, groups = build_dataset(df, problems_dict)

print("Samples:", len(X), "| Features:", X.shape[1], "| Unique Problems:", len(np.unique(groups)))

# Weights transform
if WEIGHT_MODE == "none":
    w = np.ones_like(w_raw, dtype=float)
elif WEIGHT_MODE == "raw":
    w = w_raw.astype(float)
elif WEIGHT_MODE == "sqrt":
    w = np.sqrt(np.clip(w_raw, EPS, None))
elif WEIGHT_MODE == "clip10":
    w = np.clip(w_raw, 1.0, 10.0)
else:
    raise ValueError("WEIGHT_MODE must be one of: none/raw/sqrt/clip10")

print(f"Weight mode: {WEIGHT_MODE} | min={w.min():.3f}, median={np.median(w):.3f}, max={w.max():.3f}")


In [ ]:
# %% [code]
# =========================
# Model builders + samplers
# =========================

def _sample_max_features(rng):
    # IMPORTANT: ensure numeric floats are floats, not strings -> fixes your InvalidParameterError
    options = ["sqrt", "log2", None, 0.6, 0.8]
    return options[int(rng.randint(len(options)))]

def sample_rf_params(rng):
    return {
        "n_estimators": int(rng.choice([300, 500, 800] if FAST_MODE else [500, 800, 1200])),
        "max_depth": rng.choice([None, 10, 14, 18]),
        "min_samples_leaf": int(rng.choice([1, 2, 4])),
        "min_samples_split": int(rng.choice([2, 5, 10])),
        "max_features": _sample_max_features(rng),
        "bootstrap": bool(rng.choice([True, False])),
    }

def build_rf(params):
    return RandomForestRegressor(
        random_state=SEED,
        n_jobs=-1,
        **params
    )

def sample_et_params(rng):
    return {
        "n_estimators": int(rng.choice([400, 700, 1000] if FAST_MODE else [700, 1000, 1400])),
        "max_depth": rng.choice([None, 12, 16, 20]),
        "min_samples_leaf": int(rng.choice([1, 2, 4])),
        "min_samples_split": int(rng.choice([2, 5, 10])),
        "max_features": _sample_max_features(rng),
    }

def build_et(params):
    return ExtraTreesRegressor(
        random_state=SEED,
        n_jobs=-1,
        **params
    )

def sample_mlp_params(rng):
    hidden_options = [(64, 32), (128, 64), (128, 64, 32), (256, 128)]
    return {
        "hidden_layer_sizes": hidden_options[int(rng.randint(len(hidden_options)))],
        "alpha": float(rng.choice([1e-6, 1e-5, 1e-4, 1e-3])),
        "learning_rate_init": float(rng.choice([3e-4, 5e-4, 1e-3, 2e-3])),
        "batch_size": int(rng.choice([32, 64, 128])),
        "max_iter": int(rng.choice([220, 300, 380] if FAST_MODE else [300, 450, 650])),
    }

def build_mlp(params):
    return MLPRegressor(
        random_state=SEED,
        activation="relu",
        solver="adam",
        early_stopping=True,
        n_iter_no_change=12,
        validation_fraction=0.1,
        **params
    )

def sample_xgb_params(rng):
    # μικρό αλλά δυνατό search space (καλό MAE χωρίς να αργεί πολύ)
    return {
        "n_estimators": int(rng.choice([900, 1200, 1600, 2000])),
        "max_depth": int(rng.choice([4, 5, 6, 7])),         # συμπεριέλαβα 4-6 για λιγότερο overfit
        "learning_rate": float(rng.choice([0.008, 0.01, 0.012, 0.015, 0.02])),
        "subsample": float(rng.choice([0.75, 0.8, 0.9, 1.0])),
        "colsample_bytree": float(rng.choice([0.75, 0.8, 0.9, 1.0])),
        "min_child_weight": float(rng.choice([2.0, 3.0, 5.0, 8.0])),
        "reg_lambda": float(rng.choice([0.5, 1.0, 2.0, 5.0, 10.0])),
        "reg_alpha": float(rng.choice([0.0, 0.05, 0.1, 0.5, 1.0, 2.0])),
        "gamma": float(rng.choice([0.0, 0.05, 0.1, 0.3, 0.8])),
    }

def build_xgb(params):
    if xgb is None:
        raise ImportError("xgboost not installed. Run: pip install xgboost")

    base = {
        "random_state": SEED,
        "n_jobs": -1,
        "tree_method": "hist",
        "eval_metric": "mae",
    }

    # objective: δοκίμασε absoluteerror αν υποστηρίζεται
    # (σε μερικές εγκαταστάσεις μπορεί να χρειάζεται fallback)
    obj_try = ["reg:absoluteerror", "reg:pseudohubererror", "reg:squarederror"]
    for obj in obj_try:
        try:
            model = xgb.XGBRegressor(objective=obj, **base, **params)
            # απλά return το πρώτο "λογικό" objective
            return model
        except Exception:
            continue

    return xgb.XGBRegressor(objective="reg:squarederror", **base, **params)


In [ ]:
# %% [code]
# =========================
# Training helpers (weights + XGB early stopping)
# =========================

def fit_model(model, Xtr, ytr, wtr, Xval=None, yval=None, wval=None):
    mod = model.__class__.__module__

    if mod.startswith("xgboost"):
        fit_kwargs = {"sample_weight": wtr, "verbose": False}
        if Xval is not None and yval is not None:
            fit_kwargs["eval_set"] = [(Xval, yval)]
            fit_kwargs["early_stopping_rounds"] = EARLY_STOPPING_ROUNDS
            # Some versions support this, some don't -> safe try
            if wval is not None:
                fit_kwargs["sample_weight_eval_set"] = [wval]
        try:
            model.fit(Xtr, ytr, **fit_kwargs)
        except TypeError:
            # fallback if sample_weight_eval_set not supported
            fit_kwargs.pop("sample_weight_eval_set", None)
            try:
                model.fit(Xtr, ytr, **fit_kwargs)
            except TypeError:
                # fallback if early_stopping_rounds not supported
                fit_kwargs.pop("early_stopping_rounds", None)
                fit_kwargs.pop("eval_set", None)
                model.fit(Xtr, ytr, sample_weight=wtr)
        return model

    # sklearn trees
    try:
        model.fit(Xtr, ytr, sample_weight=wtr)
    except TypeError:
        model.fit(Xtr, ytr)
    return model

def cv_mae(build_fn, sample_fn, X, y, w, groups, splits, scale=False, n_trials=10, model_name=""):
    rng = np.random.RandomState(SEED)
    best = {"mae": 1e9, "params": None}

    Xv = X.values

    for _ in range(n_trials):
        params = sample_fn(rng)

        fold_maes = []
        for tr_i, val_i in splits:
            Xtr, Xval = Xv[tr_i], Xv[val_i]
            ytr, yval = y[tr_i], y[val_i]
            wtr, wval = w[tr_i], w[val_i]

            if scale:
                sc = StandardScaler()
                Xtr = sc.fit_transform(Xtr)
                Xval = sc.transform(Xval)

            model = build_fn(params)
            model = fit_model(model, Xtr, ytr, wtr, Xval=Xval, yval=yval, wval=wval)
            pred = clip01(model.predict(Xval))
            fold_maes.append(mean_absolute_error(yval, pred))

        mae = float(np.mean(fold_maes))
        if mae < best["mae"]:
            best = {"mae": mae, "params": params}

    return best

def cv_oof(build_fn, params, X, y, w, groups, splits, scale=False):
    Xv = X.values
    oof = np.zeros_like(y, dtype=float)

    for tr_i, val_i in splits:
        Xtr, Xval = Xv[tr_i], Xv[val_i]
        ytr, yval = y[tr_i], y[val_i]
        wtr, wval = w[tr_i], w[val_i]

        sc = None
        if scale:
            sc = StandardScaler()
            Xtr = sc.fit_transform(Xtr)
            Xval = sc.transform(Xval)

        model = build_fn(params)
        model = fit_model(model, Xtr, ytr, wtr, Xval=Xval, yval=yval, wval=wval)
        oof[val_i] = model.predict(Xval)

    return oof

def cv_oof_xgb_bagged(params, X, y, w, groups, splits, seeds):
    # bagging μόνο για XGB (2 seeds fast)
    Xv = X.values
    oof = np.zeros_like(y, dtype=float)

    for tr_i, val_i in splits:
        Xtr, Xval = Xv[tr_i], Xv[val_i]
        ytr, yval = y[tr_i], y[val_i]
        wtr, wval = w[tr_i], w[val_i]

        pred_sum = np.zeros(len(val_i), dtype=float)
        for rs in seeds:
            model = build_xgb(params)
            try:
                model.set_params(random_state=int(rs))
            except Exception:
                pass
            model = fit_model(model, Xtr, ytr, wtr, Xval=Xval, yval=yval, wval=wval)
            pred_sum += model.predict(Xval)

        oof[val_i] = pred_sum / float(len(seeds))

    return oof


In [ ]:
# %% [code]
# =========================
# CV Splits
# =========================
tune_splits  = list(GroupKFold(n_splits=TUNE_SPLITS).split(np.zeros(len(y)), y, groups=groups))
final_splits = list(GroupKFold(n_splits=FINAL_SPLITS).split(np.zeros(len(y)), y, groups=groups))

print("Tune folds:", TUNE_SPLITS, "| Final folds:", FINAL_SPLITS)
print("xgboost available:", xgb is not None)


In [ ]:
# %% [code]
# =========================
# Model registry (4 models)
# =========================
if xgb is None:
    raise ImportError("xgboost is required for this pipeline. Install: pip install xgboost")

MODELS = {
    "XGBoost":      {"build": build_xgb, "sample": sample_xgb_params, "scale": False, "trials": TRIALS_XGB},
    "RandomForest": {"build": build_rf,  "sample": sample_rf_params,  "scale": False, "trials": TRIALS_RF},
    "ExtraTrees":   {"build": build_et,  "sample": sample_et_params,  "scale": False, "trials": TRIALS_ET},
    "MLP":          {"build": build_mlp, "sample": sample_mlp_params, "scale": True,  "trials": TRIALS_MLP},
}


In [ ]:
# %% [code]
# =========================
# Stage A: FAST tuning (3-fold for all)
# =========================
tuned = {}

for name, cfg in MODELS.items():
    best = cv_mae(
        build_fn=cfg["build"],
        sample_fn=cfg["sample"],
        X=X, y=y, w=w, groups=groups,
        splits=tune_splits,
        scale=cfg["scale"],
        n_trials=cfg["trials"],
        model_name=name
    )
    tuned[name] = best
    print(f"{name}: tune_mae={best['mae']:.6f} params={best['params']}")


In [ ]:
# %% [code]
# =========================
# Stage B: FINAL eval (5-fold μόνο για top2)
# =========================
# Επιλέγουμε top2 από tune_mae
ranked = sorted([(k, v["mae"]) for k, v in tuned.items()], key=lambda x: x[1])
top2_names = [ranked[0][0], ranked[1][0]]
print("Top-2 after tuning:", top2_names)

results = []
oof_preds = {}

for name in MODELS.keys():
    cfg = MODELS[name]
    params = tuned[name]["params"]

    if name in top2_names:
        # final 5-fold
        if name == "XGBoost":
            oof = cv_oof_xgb_bagged(params, X, y, w, groups, final_splits, seeds=XGB_OOF_SEEDS)
        else:
            oof = cv_oof(cfg["build"], params, X, y, w, groups, final_splits, scale=cfg["scale"])
        mae, rmse, r2 = metrics(y, oof)
        results.append((name, mae, rmse, r2, "FINAL_5fold"))
        oof_preds[name] = oof
    else:
        # γρήγορο 3-fold OOF για τα υπόλοιπα (μόνο για ranking, όχι τελικό)
        if name == "XGBoost":
            oof = cv_oof_xgb_bagged(params, X, y, w, groups, tune_splits, seeds=[SEED])
        else:
            oof = cv_oof(cfg["build"], params, X, y, w, groups, tune_splits, scale=cfg["scale"])
        mae, rmse, r2 = metrics(y, oof)
        results.append((name, mae, rmse, r2, "TUNE_3fold"))
        oof_preds[name] = oof

res_df = pd.DataFrame(results, columns=["Model", "MAE", "RMSE", "R2", "Eval"])
res_df = res_df.sort_values(["Eval","MAE"], ascending=[True, True]).reset_index(drop=True)
display(res_df)

# Απόφαση: καλύτερο μοντέλο με βάση το FINAL_5fold (αν υπάρχει), αλλιώς το μικρότερο MAE overall
final_candidates = res_df[res_df["Eval"]=="FINAL_5fold"].sort_values("MAE")
if len(final_candidates) > 0:
    best_model = final_candidates.iloc[0]["Model"]
    best_mae = float(final_candidates.iloc[0]["MAE"])
else:
    best_model = res_df.iloc[0]["Model"]
    best_mae = float(res_df.iloc[0]["MAE"])

print(f"Best model selected: {best_model} | MAE={best_mae:.6f}")


In [ ]:
# %% [code]
# =========================
# Train final model on ALL data
# =========================
best_cfg = MODELS[best_model]
best_params = tuned[best_model]["params"]

# Prepare full X (with scaling for MLP)
X_full = X.values
scaler = None
if best_cfg["scale"]:
    scaler = StandardScaler()
    X_full = scaler.fit_transform(X_full)

final_model = best_cfg["build"](best_params)
final_model = fit_model(final_model, X_full, y, w)

print("Final training complete.")


In [ ]:
# %% [code]
# =========================
# Fast feature importance plot (only for tree models with feature_importances_)
# =========================
if hasattr(final_model, "feature_importances_"):
    imp = np.asarray(final_model.feature_importances_, dtype=float)
    order = np.argsort(imp)[::-1][:15]
    top_feats = X.columns[order]
    top_scores = imp[order]

    plt.figure(figsize=(9,5))
    plt.barh(top_feats[::-1], top_scores[::-1])
    plt.title(f"{best_model} feature importance")
    plt.tight_layout()
    plt.show()

    fi_df = pd.DataFrame({"Feature": top_feats, "Score": top_scores})
    display(fi_df)
else:
    print("No built-in feature_importances_ for this model (MLP).")
